In [ ]:
'''
Submitted by: Criselvi lindsey
Class: DSC640-T301 WK1&2 Milestone 3
Due Date: 04/22/2026
'''

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import json
import urllib.request

cols_needed = [
    "State_Name",
    "State_Abbreviation",
    "County_Name",
    "County_FIPS_Code",
    "StudyYear",
    "MCInfant",
    "MCToddler",
    "MCPreschool",
    "MFCCInfant",
    "_75CInfant",
    "MHI",
    "H_Under6_BothWork",
    "H_Under6_SingleM",
    "TotalPop"
]

df = pd.read_excel(
    "nationaldatabaseofchildcareprices.xlsx",
    usecols=cols_needed
)

# Preview data
print(df.head())
print(df.columns.tolist())
# df.columns

In [ ]:
# Number of unique years
print("Unique Study Years:", df["StudyYear"].nunique())

# List of unique years
print("Years in Dataset:")
print(sorted(df["StudyYear"].unique()))

print("Rows per year:")
print(df["StudyYear"].value_counts().sort_index())

In [ ]:
# print the value.counts



In [ ]:
# WEEKLY COST TREND BY YEAR

weekly_trend = (
    df.groupby("StudyYear")
    .agg(
        Avg_Weekly_Infant_Cost=("MCInfant", "mean"),
        Avg_Weekly_Toddler_Cost=("MCToddler", "mean"),
        Avg_Weekly_Preschool_Cost=("MCPreschool", "mean")
    )
    .reset_index()
)

print(weekly_trend)

In [ ]:
plt.figure(figsize=(10,6))

sns.lineplot(
    data=weekly_trend,
    x="StudyYear",
    y="Avg_Weekly_Infant_Cost",
    marker="o",
    label="Infant"
)

sns.lineplot(
    data=weekly_trend,
    x="StudyYear",
    y="Avg_Weekly_Toddler_Cost",
    marker="o",
    label="Toddler"
)

sns.lineplot(
    data=weekly_trend,
    x="StudyYear",
    y="Avg_Weekly_Preschool_Cost",
    marker="o",
    label="Preschool"
)

plt.title("Average Weekly Childcare Cost by Year")
plt.xlabel("Study Year")
plt.ylabel("Average Weekly Cost ($)")
plt.show()

In [ ]:
# Preview monthly cost columns
df["Monthly_Infant_Cost"] = df["MCInfant"] * 52 / 12
df["Monthly_Toddler_Cost"] = df["MCToddler"] * 52 / 12
df["Monthly_Preschool_Cost"] = df["MCPreschool"] * 52 / 12
monthly_trend = (
    df.groupby("StudyYear")
    .agg(
        Avg_Monthly_Infant_Cost=("Monthly_Infant_Cost", "mean"),
        Avg_Monthly_Toddler_Cost=("Monthly_Toddler_Cost", "mean"),
        Avg_Monthly_Preschool_Cost=("Monthly_Preschool_Cost", "mean")
    )
    .reset_index()
)

print(monthly_trend)


In [ ]:
# Annual Trend

# CREATE ANNUAL COST COLUMNS
# -----------------------------
df["Annual_Infant_Cost"] = df["MCInfant"] * 52
df["Annual_Toddler_Cost"] = df["MCToddler"] * 52
df["Annual_Preschool_Cost"] = df["MCPreschool"] * 52

annual_trend = (
    df.groupby("StudyYear")
    .agg(
        Avg_Annual_Infant_Cost=("Annual_Infant_Cost", "mean"),
        Avg_Annual_Toddler_Cost=("Annual_Toddler_Cost", "mean"),
        Avg_Annual_Preschool_Cost=("Annual_Preschool_Cost", "mean")
    )
    .reset_index()
)

print(annual_trend)


In [ ]:
# Annual cost difference
df["Annual_Cost_Difference"] = (
    df["Annual_Infant_Cost"] - df["Annual_Toddler_Cost"]
)

# Top 10 county-year records with largest annual infant vs toddler gap
top_annual_diff = df[
    [
        "StudyYear",
        "State_Name",
        "County_Name",
        "Annual_Infant_Cost",
        "Annual_Toddler_Cost",
        "Annual_Cost_Difference"
    ]
].sort_values(
    by="Annual_Cost_Difference",
    ascending=False
).head(10)

print(top_annual_diff)

In [ ]:
top_counties_avg_gap = (
    df.groupby(["State_Name", "County_Name"])
    .agg(
        Avg_Annual_Cost_Difference=("Annual_Cost_Difference", "mean")
    )
    .reset_index()
    .sort_values(by="Avg_Annual_Cost_Difference", ascending=False)
    .head(10)
)

print(top_counties_avg_gap)

In [ ]:
top_counties_with_volatility = (
    df.groupby(["State_Name", "County_Name"])
    .agg(
        Avg_Annual_Cost_Difference=("Annual_Cost_Difference", "mean"),
        Std_Annual_Cost_Difference=("Annual_Cost_Difference", "std")
    )
    .reset_index()
    .sort_values(by="Avg_Annual_Cost_Difference", ascending=False)
    .head(10)
)

print(top_counties_with_volatility)

In [ ]:
county_obs = (
    df.groupby(["State_Name", "County_Name"])
    .size()
    .reset_index(name="Observation_Count")
)

print(
    county_obs[
        county_obs["County_Name"].isin(["Richmond County", "Bronx County"])
    ]
)

In [ ]:
print(
    df[
        (df["State_Name"] == "New York") &
        (df["County_Name"] == "Richmond County")
    ][
        ["StudyYear", "Annual_Cost_Difference"]
    ]
)

print(
    df[
        (df["State_Name"] == "New York") &
        (df["County_Name"] == "Bronx County")
    ][
        ["StudyYear", "Annual_Cost_Difference"]
    ]
)

In [ ]:
county_gap_stats = (
    df.groupby(["County_FIPS_Code", "State_Name", "County_Name"])
    .agg(
        Avg_Annual_Cost_Difference=("Annual_Cost_Difference", "mean"),
        Std_Annual_Cost_Difference=("Annual_Cost_Difference", "std"),
        Valid_Observation_Count=("Annual_Cost_Difference", "count")
    )
    .reset_index()
)

filtered_top = (
    county_gap_stats[
        county_gap_stats["Valid_Observation_Count"] >= 5
    ]
    .sort_values(by="Avg_Annual_Cost_Difference", ascending=False)
    .head(10)
)

print(filtered_top)

In [ ]:
county_gap_stats["Consistency_Score"] = (
    county_gap_stats["Avg_Annual_Cost_Difference"] /
    county_gap_stats["Std_Annual_Cost_Difference"]
)

In [ ]:
# Merge county-level consistency metrics onto each county-year row
df = df.merge(
    county_gap_stats[
        [
            "County_FIPS_Code",
            "Avg_Annual_Cost_Difference",
            "Std_Annual_Cost_Difference",
            "Valid_Observation_Count",
            "Consistency_Score",
        ]
    ],
    on="County_FIPS_Code",
    how="left"
)

"Avg_Annual_Cost_Difference",
"Std_Annual_Cost_Difference",
"Valid_Observation_Count",
"Consistency_Score",



In [ ]:
final_cols = [
    "StudyYear",
    "State_Name",
    "State_Abbreviation",
    "County_Name",
    "County_FIPS_Code",

    # Weekly costs
    "MCInfant",
    "MCToddler",
    "MCPreschool",

    # Monthly costs
    "Monthly_Infant_Cost",
    "Monthly_Toddler_Cost",
    "Monthly_Preschool_Cost",

    # Annual costs
    "Annual_Infant_Cost",
    "Annual_Toddler_Cost",
    "Annual_Preschool_Cost",

    # Difference metrics
    "Annual_Cost_Difference",

    # Affordability
    "MHI",
    "Infant_Cost_Pct_Income",

    # County-level gap metrics
    "Avg_Annual_Cost_Difference",
    "Std_Annual_Cost_Difference",
    "Valid_Observation_Count",
    "Consistency_Score"
]
df["Infant_Cost_Pct_Income"] = (
    df["Annual_Infant_Cost"] / df["MHI"]
) * 100

df_clean = df[final_cols].copy()

numeric_cols = df_clean.select_dtypes(include="number").columns
df_clean[numeric_cols] = df_clean[numeric_cols].round(2)

df_clean.to_csv("childcare_cleaned_for_pbi_trend.csv", index=False)

print("Export complete: childcare_cleaned_for_pbi_trend.csv")

In [ ]:
# -----------------------------
# YEARLY TREND: INFANT COST % OF INCOME
# -----------------------------

affordability_trend = (
    df.groupby("StudyYear")
    .agg(
        Avg_Annual_Infant_Cost=("Annual_Infant_Cost", "mean"),
        Avg_MHI=("MHI", "mean")
    )
    .reset_index()
)

affordability_trend["Avg_Infant_Cost_Pct_Income"] = (
    affordability_trend["Avg_Annual_Infant_Cost"] /
    affordability_trend["Avg_MHI"]
) * 100


# -----------------------------
# ACCESSIBLE LINE CHART WITH DATA LABELS
# -----------------------------

plt.figure(figsize=(11, 6))

ax = sns.lineplot(
    data=affordability_trend,
    x="StudyYear",
    y="Avg_Infant_Cost_Pct_Income",
    marker="o",
    linewidth=2.5,
    markersize=7,
    color="#0072B2"
)

# Add percentage labels above each point
for x, y in zip(
    affordability_trend["StudyYear"],
    affordability_trend["Avg_Infant_Cost_Pct_Income"]
):
    plt.text(
        x,
        y + 0.15,
        f"{y:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.title(
    "Average Infant Childcare Cost Burden Over Time",
    fontsize=15,
    weight="bold",
    loc="left",
    pad=15
)

plt.xlabel("Study Year", fontsize=11, labelpad=10)
plt.ylabel("Infant Cost as % of Median Household Income", fontsize=11, labelpad=10)

plt.xticks(
    ticks=affordability_trend["StudyYear"],
    labels=affordability_trend["StudyYear"]
)

plt.ylim(10, 20)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()

plt.show()

In [ ]:
map_year = 2018

coverage_pct = (
    df[
        (df["StudyYear"] == map_year) &
        (df["Infant_Cost_Pct_Income"].notna())
    ]["County_FIPS_Code"].count()
    /
    df[df["StudyYear"] == map_year]["County_FIPS_Code"].count()
) * 100

print(f"Coverage for {map_year}: {coverage_pct:.1f}%")

In [ ]:
# -----------------------------
# SETTINGS
# -----------------------------
map_year = 2018

geojson_url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"

# Load county GeoJSON
with urllib.request.urlopen(geojson_url) as response:
    counties_geojson = json.load(response)

# Create list of ALL U.S. county FIPS codes from GeoJSON
all_counties = pd.DataFrame({
    "County_FIPS_Code": [
        feature["id"] for feature in counties_geojson["features"]
    ]
})

# PREPARE  THE DATA

df_map = df[df["StudyYear"] == map_year].copy()

df_map["County_FIPS_Code"] = (
    df_map["County_FIPS_Code"]
    .astype(str)
    .str.zfill(5)
)

# Keep only needed columns
df_map = df_map[
    [
        "County_FIPS_Code",
        "County_Name",
        "State_Name",
        "Infant_Cost_Pct_Income"
    ]
]

# Merge  data onto ALL counties
df_full_map = all_counties.merge(
    df_map,
    on="County_FIPS_Code",
    how="left"
)


# CREATE AFFORDABILITY BANDS

band_order = ["<10%", "10–15%", "15–20%", "20–25%", "25%+", "Missing"]

df_full_map["Affordability_Band"] = pd.cut(
    df_full_map["Infant_Cost_Pct_Income"],
    bins=[0, 10, 15, 20, 25, 100],
    labels=["<10%", "10–15%", "15–20%", "20–25%", "25%+"],
    include_lowest=True
)

df_full_map["Affordability_Band"] = (
    df_full_map["Affordability_Band"]
    .astype("object")
    .fillna("Missing")
)

# COMPLETE COUNTY MAP

fig = px.choropleth(
    df_full_map,
    geojson=counties_geojson,
    locations="County_FIPS_Code",
    color="Affordability_Band",
    category_orders={"Affordability_Band": band_order},
    color_discrete_map={
        "<10%": "#deebf7",
        "10–15%": "#9ecae1",
        "15–20%": "#6baed6",
        "20–25%": "#3182bd",
        "25%+": "#08519c",
        "Missing": "#d9d9d9"
    },
    scope="usa",
    hover_data={
        "County_Name": True,
        "State_Name": True,
        "Infant_Cost_Pct_Income": ":.1f"
    },
    title=f"County-Level Infant Childcare Cost Burden ({map_year})"
)

fig.update_geos(visible=False)

fig.update_layout(
    title_x=0.01,
    legend_title_text="Infant Cost as % of Income",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

In [ ]:
# PREPARE INDEXED TREND DATA

indexed_trend = (
    df.groupby("StudyYear")
    .agg(
        Avg_Annual_Infant_Cost=("Annual_Infant_Cost", "mean"),
        Avg_MHI=("MHI", "mean")
    )
    .reset_index()
)

# Index both series to base year = 2008
base_cost = indexed_trend.loc[
    indexed_trend["StudyYear"] == 2008,
    "Avg_Annual_Infant_Cost"
].iloc[0]

base_income = indexed_trend.loc[
    indexed_trend["StudyYear"] == 2008,
    "Avg_MHI"
].iloc[0]

indexed_trend["Infant_Cost_Index"] = (
    indexed_trend["Avg_Annual_Infant_Cost"] / base_cost
) * 100

indexed_trend["Income_Index"] = (
    indexed_trend["Avg_MHI"] / base_income
) * 100

# -----------------------------
# ACCESSIBLE MULTI-LINE CHART
# -----------------------------
plt.figure(figsize=(11, 6))

sns.lineplot(
    data=indexed_trend,
    x="StudyYear",
    y="Infant_Cost_Index",
    marker="o",
    linewidth=2.5,
    color="#0072B2",
    label="Infant Childcare Cost"
)

sns.lineplot(
    data=indexed_trend,
    x="StudyYear",
    y="Income_Index",
    marker="o",
    linewidth=2.5,
    color="#E69F00",
    label="Median Household Income"
)

plt.title(
    "Growth in Infant Childcare Costs vs Household Income (2008 = 100)",
    fontsize=15,
    weight="bold",
    loc="left",
    pad=15
)

plt.xlabel("Study Year", fontsize=11, labelpad=10)
plt.ylabel("Indexed Growth (2008 = 100)", fontsize=11, labelpad=10)

plt.xticks(indexed_trend["StudyYear"])

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()

plt.show()

In [ ]:
# -----------------------------
# SET AFFORDABILITY THRESHOLD
# -----------------------------
threshold = 15

# -----------------------------
# CALCULATE % OF COUNTIES ABOVE THRESHOLD BY YEAR
# -----------------------------
threshold_trend = (
    df[df["Infant_Cost_Pct_Income"].notna()]
    .groupby("StudyYear")
    .agg(
        Total_Counties=("County_FIPS_Code", "count"),
        Counties_Above_Threshold=(
            "Infant_Cost_Pct_Income",
            lambda x: (x > threshold).sum()
        )
    )
    .reset_index()
)

threshold_trend["Pct_Counties_Above_Threshold"] = (
    threshold_trend["Counties_Above_Threshold"] /
    threshold_trend["Total_Counties"]
) * 100

print(threshold_trend)

# -----------------------------
# ACCESSIBLE LINE CHART
# -----------------------------
plt.figure(figsize=(11, 6))

sns.lineplot(
    data=threshold_trend,
    x="StudyYear",
    y="Pct_Counties_Above_Threshold",
    marker="o",
    linewidth=2.5,
    markersize=7,
    color="#D55E00"   # color-blind friendly orange/red
)

# Add labels
for x, y in zip(
    threshold_trend["StudyYear"],
    threshold_trend["Pct_Counties_Above_Threshold"]
):
    plt.text(
        x,
        y + 1,
        f"{y:.1f}%",
        ha="center",
        fontsize=9
    )

plt.title(
    f"Percent of Counties Where Infant Childcare Exceeds {threshold}% of Income",
    fontsize=15,
    weight="bold",
    loc="left",
    pad=15
)

plt.xlabel("Study Year", fontsize=11, labelpad=10)
plt.ylabel("Percent of Counties Above Threshold", fontsize=11, labelpad=10)

plt.xticks(threshold_trend["StudyYear"])

plt.ylim(0, 100)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()

plt.show()

In [ ]:
# -----------------------------
# FILTER TO VALID RECORDS
# -----------------------------
df_workforce = df[
    df["Infant_Cost_Pct_Income"].notna() &
    df["H_Under6_BothWork"].notna() &
    df["TotalPop"].notna()
].copy()

# Optional: focus on latest year for cleaner story
df_workforce = df_workforce[df_workforce["StudyYear"] == 2018]

# -----------------------------
# SCATTERPLOT
# -----------------------------
plt.figure(figsize=(11, 6))

sns.scatterplot(
    data=df_workforce,
    x="H_Under6_BothWork",
    y="Infant_Cost_Pct_Income",
    size="TotalPop",
    sizes=(20, 250),
    alpha=0.6,
    color="#0072B2"
)

plt.title(
    "Infant Childcare Burden vs Dual-Working Households with Young Children",
    fontsize=15,
    weight="bold",
    loc="left",
    pad=15
)

plt.xlabel(
    "Households with Children Under 6 Where Both Parents Work",
    fontsize=11,
    labelpad=10
)

plt.ylabel(
    "Infant Childcare Cost as % of Median Household Income",
    fontsize=11,
    labelpad=10
)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", alpha=0.10)

sns.despine()
plt.tight_layout()

plt.show()